In [1]:
!git clone https://github.com/Anand-786/llm-quantization-thesis.git
%cd /content/llm-quantization-thesis
!git clone https://github.com/mit-han-lab/smoothquant.git smoothquant_repo
!pip uninstall smoothquant -y
!cd smoothquant_repo && pip install -e .
!pip install -q transformers accelerate datasets zstandard tqdm sentencepiece

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
SAVE_DIR = "/content/drive/MyDrive/thesis_results/verification/llama-2-13b"
os.makedirs(SAVE_DIR, exist_ok=True)

DRIVE_SCALES = "/content/drive/MyDrive/thesis_results/act_scales/llama-2-13b.pt"
REPO_SCALES  = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/llama-2-13b.pt"
assert os.path.exists(DRIVE_SCALES), f"missing: {DRIVE_SCALES} — run generate_act_scales_cells.md first."
os.makedirs(os.path.dirname(REPO_SCALES), exist_ok=True)
shutil.copy2(DRIVE_SCALES, REPO_SCALES)

print("max scales :", REPO_SCALES)
!nvidia-smi

Cloning into 'llm-quantization-thesis'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 251 (delta 105), reused 211 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (251/251), 5.31 MiB | 28.74 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/llm-quantization-thesis
Cloning into 'smoothquant_repo'...
remote: Enumerating objects: 352, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 352 (delta 120), reused 90 (delta 90), pack-reused 183 (from 1)
Receiving objects: 100% (352/352), 6.80 MiB | 29.75 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Obtaining file:///content/llm-quantization-thesis/smoothquant_repo
  Preparing metadata (setup.py) ... done
  Running setup.py develop for smoothquant
Mounted at /content/drive
max scales : /content/llm-quantization-thesis/smoothquant_repo/act_scale

In [8]:
import re, torch, json

SCALES_PATH = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/llama-2-13b.pt"
SAVE_DIR    = "/content/drive/MyDrive/thesis_results/verification/llama-2-13b"
ALPHA_MIN, ALPHA_MAX = 0.80, 0.95   # focused range — anchored near paper's Llama-2 optimum (α=0.85)

raw = torch.load(SCALES_PATH, map_location="cpu")
LAYER_RE = re.compile(r"model\.layers\.(\d+)\.(.+)")

sev_by_layer = {}
for name, vec in raw.items():
    m = LAYER_RE.match(name)
    if not m:
        continue
    suffix = m.group(2)
    if suffix not in ("self_attn.q_proj", "mlp.gate_proj"):
        continue
    layer = int(m.group(1))
    v = vec.float().abs()
    s = (v.max() / v.median().clamp(min=1e-12)).item()
    sev_by_layer.setdefault(layer, []).append(s)

layers = sorted(sev_by_layer.keys())
sev = torch.tensor([sum(sev_by_layer[l]) / len(sev_by_layer[l]) for l in layers])

# Linear normalisation — original OPT-style sev/sev_max.
sev_norm = sev / sev.max()
alpha_per_layer = (ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * sev_norm).tolist()

assert len(alpha_per_layer) == 40, f"expected 40 layers, got {len(alpha_per_layer)}"
print(f"per-layer α range: [{min(alpha_per_layer):.3f}, {max(alpha_per_layer):.3f}]  (40 layers)")
print(f"severity spread (max/min): {sev.max().item() / sev.min().item():.2f}×")
print()
print("layer  severity   α(l)")
for l, s, a in zip(layers, sev.tolist(), alpha_per_layer):
    print(f"  {l:2d}    {s:7.2f}    {a:.3f}")

# Persist the schedule so it survives runtime disconnects.
with open(f"{SAVE_DIR}/llama-2-13b_alpha_schedule.json", "w") as f:
    json.dump({
        "layers": layers,
        "severity": sev.tolist(),
        "alpha_per_layer": alpha_per_layer,
        "alpha_range": [ALPHA_MIN, ALPHA_MAX],
        "normalisation": "linear",
    }, f, indent=2)
print(f"\nschedule saved -> {SAVE_DIR}/llama-2-13b_alpha_schedule.json")

per-layer α range: [0.805, 0.950]  (40 layers)
severity spread (max/min): 29.52×

layer  severity   α(l)
   0     180.13    0.950
   1      31.47    0.826
   2      11.65    0.810
   3      16.69    0.814
   4       8.14    0.807
   5       9.16    0.808
   6      10.00    0.808
   7      16.24    0.814
   8      10.71    0.809
   9      11.09    0.809
  10      13.20    0.811
  11      16.17    0.813
  12      15.69    0.813
  13      13.12    0.811
  14      15.04    0.813
  15      12.99    0.811
  16      11.25    0.809
  17      10.23    0.809
  18       9.63    0.808
  19      10.25    0.809
  20       9.78    0.808
  21      10.41    0.809
  22       9.06    0.808
  23       8.13    0.807
  24       7.45    0.806
  25       8.36    0.807
  26       7.14    0.806
  27       6.46    0.805
  28       6.28    0.805
  29       6.81    0.806
  30       6.10    0.805
  31       6.20    0.805
  32       6.63    0.806
  33       7.22    0.806
  34       7.18    0.806
  35       7.26    0

In [9]:
%%writefile /content/smooth_per_layer_llama.py
"""Per-layer α extension of smoothquant.smooth.smooth_lm (Llama)."""
import re
import torch
from transformers.models.llama.modeling_llama import LlamaDecoderLayer
from smoothquant.smooth import smooth_ln_fcs_llama_like

LAYER_RE = re.compile(r"model\.layers\.(\d+)$")

@torch.no_grad()
def smooth_lm_per_layer_llama(model, scales, alpha_schedule):
    if not isinstance(alpha_schedule, (list, tuple)):
        raise TypeError("alpha_schedule must be a list/tuple of floats")
    for name, module in model.named_modules():
        if not isinstance(module, LlamaDecoderLayer):
            continue
        m = LAYER_RE.match(name)
        if m is None:
            continue
        layer_idx = int(m.group(1))
        alpha = float(alpha_schedule[layer_idx])

        attn_ln = module.input_layernorm
        qkv = [module.self_attn.q_proj, module.self_attn.k_proj, module.self_attn.v_proj]
        qkv_input_scales = scales[name + ".self_attn.q_proj"]
        smooth_ln_fcs_llama_like(attn_ln, qkv, qkv_input_scales, alpha)

        ffn_ln = module.post_attention_layernorm
        fcs = [module.mlp.gate_proj, module.mlp.up_proj]
        fcs_input_scales = scales[name + ".mlp.gate_proj"]
        smooth_ln_fcs_llama_like(ffn_ln, fcs, fcs_input_scales, alpha)

Overwriting /content/smooth_per_layer_llama.py


In [10]:
import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/llm-quantization-thesis/smoothquant_repo")

import torch, torch.nn as nn, json, time, tqdm, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from smoothquant.smooth import smooth_lm
from smoothquant.fake_quant import quantize_model
from smooth_per_layer_llama import smooth_lm_per_layer_llama

MODEL = "NousResearch/Llama-2-13b-hf"
SCALES_PATH = "/content/llm-quantization-thesis/smoothquant_repo/act_scales/llama-2-13b.pt"
SAVE_DIR    = "/content/drive/MyDrive/thesis_results/verification/llama-2-13b"


class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=40):
        self.dataset = tokenizer("\n\n".join(dataset["text"]), return_tensors="pt").input_ids.to(device)
        self.n_samples = n_samples
    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        nlls = []
        n = self.n_samples
        for i in tqdm.tqdm(range(n), desc="PPL"):
            batch = self.dataset[:, (i * 2048):((i + 1) * 2048)].to(model.device)
            logits = model(batch).logits
            shift_logits = logits[:, :-1, :].contiguous().float()
            shift_labels = self.dataset[:, (i * 2048):((i + 1) * 2048)][:, 1:]
            loss = nn.CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            nlls.append(loss.float() * 2048)
        return torch.exp(torch.stack(nlls).sum() / (n * 2048))


print("Loading tokenizer + dataset + scales...")
# NOTE: NousResearch/Llama-2-13b-hf ships a broken tokenizer.json for transformers v5.
# Llama-2's tokenizer is identical across 7B/13B/70B, so we load it from the 7B mirror.
tokenizer  = AutoTokenizer.from_pretrained("NousResearch/Llama-2-7b-hf", use_fast=True)
dataset    = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
evaluator  = Evaluator(dataset, tokenizer, "cuda")
act_scales = torch.load(SCALES_PATH)

C_QPARAMS   = dict(weight_quant="per_channel", act_quant="per_token", quantize_bmm_input=True)
PAPER_ALPHA = 0.85  # SmoothQuant paper Table 7 — Llama-2-13B row

results = []  # appended to by each run cell below

def _run_config(label, smooth_spec, qparams):
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")

    if smooth_spec == "none":
        pass
    elif smooth_spec[0] == "max":
        smooth_lm(model, act_scales, smooth_spec[1])
    elif smooth_spec[0] == "perlayer":
        smooth_lm_per_layer_llama(model, act_scales, smooth_spec[1])
    else:
        raise ValueError(f"unknown smooth spec: {smooth_spec}")

    if qparams is not None:
        model = quantize_model(model, **qparams)

    ppl = evaluator.evaluate(model).item()
    elapsed = time.time() - t0
    print(f">>> {label}: PPL = {ppl:.4f}  ({elapsed:.0f}s)")

    rec = {
        "model": MODEL,
        "label": label,
        "smooth": (smooth_spec if isinstance(smooth_spec, str)
                   else (smooth_spec[0] if smooth_spec[0] != "perlayer" else "perlayer")),
        "qparams": qparams,
        "ppl": round(ppl, 4),
        "seconds": round(elapsed, 1),
    }
    results.append(rec)
    with open(f"{SAVE_DIR}/llama-2-13b_{label}.json", "w") as f:
        json.dump(rec, f, indent=2)

    del model
    torch.cuda.empty_cache()
    return rec

print("setup ready — run the three config cells below in order.")

Loading tokenizer + dataset + scales...
setup ready — run the three config cells below in order.


In [5]:
_run_config("1_FP16", "none", None)


  1_FP16


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/196 [00:00<?, ?B/s]

PPL: 100%|██████████| 40/40 [00:11<00:00,  3.50it/s]


>>> 1_FP16: PPL = 5.1828  (90s)


{'model': 'NousResearch/Llama-2-13b-hf',
 'label': '1_FP16',
 'smooth': 'none',
 'qparams': None,
 'ppl': 5.1828,
 'seconds': 90.3}

In [6]:
_run_config(f"2_C_max_a{PAPER_ALPHA}", ("max", PAPER_ALPHA), C_QPARAMS)


  2_C_max_a0.85


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

PPL: 100%|██████████| 40/40 [00:14<00:00,  2.83it/s]

>>> 2_C_max_a0.85: PPL = 5.2255  (522s)


{'model': 'NousResearch/Llama-2-13b-hf',
 'label': '2_C_max_a0.85',
 'smooth': 'max',
 'qparams': {'weight_quant': 'per_channel',
  'act_quant': 'per_token',
  'quantize_bmm_input': True},
 'ppl': 5.2255,
 'seconds': 521.5}

In [11]:
_run_config("3_C_perlayer", ("perlayer", alpha_per_layer), C_QPARAMS)


  3_C_perlayer


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

PPL: 100%|██████████| 40/40 [00:14<00:00,  2.83it/s]

>>> 3_C_perlayer: PPL = 5.2335  (520s)


{'model': 'NousResearch/Llama-2-13b-hf',
 'label': '3_C_perlayer',
 'smooth': 'perlayer',
 'qparams': {'weight_quant': 'per_channel',
  'act_quant': 'per_token',
  'quantize_bmm_input': True},
 'ppl': 5.2335,
 'seconds': 520.3}

In [12]:
def ppl(label):
    return next(r for r in results if r["label"] == label)["ppl"]

PAPER_FP16, PAPER_SQ, PAPER_ALPHA = 4.950, 4.929, 0.85
our_fp16    = ppl("1_FP16")
our_paper   = ppl(f"2_C_max_a{PAPER_ALPHA}")
our_alpha_l = ppl("3_C_perlayer")

print(f"\n{'='*78}")
print(f"  Llama-2-13B — WikiText-2 PPL — paper-comparable table")
print(f"{'='*78}")
print(f"\n{'config':<30} {'ours':>10} {'paper':>10} {'Δ ours−paper':>14}")
print("-" * 70)
print(f"{'FP16':<30} {our_fp16:>10.4f} {PAPER_FP16:>10.4f} {our_fp16 - PAPER_FP16:>+14.4f}")
print(f"{'C + max α=0.85 (paper cfg)':<30} {our_paper:>10.4f} {PAPER_SQ:>10.4f} {our_paper - PAPER_SQ:>+14.4f}")
print(f"{'C + per-layer α (ours)':<30} {our_alpha_l:>10.4f} {'—':>10} {'—':>14}")

print(f"\nDeltas vs OUR FP16 (within-session, noise-free):")
print(f"  C max α=0.85       − FP16  = {our_paper   - our_fp16:+.4f}")
print(f"  C per-layer α      − FP16  = {our_alpha_l - our_fp16:+.4f}")
print(f"  C per-layer        − C max = {our_alpha_l - our_paper:+.4f}  (negative → α(l) wins)")

print(f"\nPaper-gap reference:")
print(f"  paper W8A8 − paper FP16    = {PAPER_SQ - PAPER_FP16:+.4f}  (Table 7)")

shift = our_fp16 - PAPER_FP16
print(f"\nProtocol-shift diagnostic:")
print(f"  our FP16 − paper FP16  = {shift:+.4f}")
if abs(shift) < 0.03:
    print(f"  → protocols match; can cite paper numbers directly.")
else:
    print(f"  → protocols differ by ~{shift:+.3f} PPL; compare *within-session* deltas only.")

# Persist combined summary (one file with all 3 rows + α schedule).
with open(f"{SAVE_DIR}/llama-2-13b_summary.json", "w") as f:
    json.dump({"results": results, "alpha_per_layer": alpha_per_layer}, f, indent=2)
print(f"\nsaved -> {SAVE_DIR}/llama-2-13b_summary.json")

StopIteration: 